In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

In [2]:
from datasets import load_dataset

sst2 = load_dataset("stanfordnlp/sst2")
class_dataset = sst2['train']['sentence'][:100]
class_targets = sst2['train']['label'][:100]

samsum = load_dataset("knkarthick/samsum")
gen_dataset = samsum['train']['dialogue'][:100]
gen_targets = samsum['train']['summary'][:100]

In [ ]:
from coolprompt.language_model.llm import DefaultLLM

raw_model = DefaultLLM.init()

[2025-07-13 17:01:52,143] [INFO] [llm.init] - Initializing default model
[2025-07-13 17:01:52,144] [DEBUG] [llm.init] - Updating default model params with langchain config: None and vllm_engine_config: None


INFO 07-13 17:02:01 __init__.py:207] Automatically detected platform cuda.
WARNING 07-13 17:02:02 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 07-13 17:02:11 config.py:549] This model supports multiple tasks: {'embed', 'classify', 'score', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 07-13 17:02:11 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='t-tech/T-lite-it-1.0', speculative_config=None, tokenizer='t-tech/T-lite-it-1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 07-13 17:03:45 model_runner.py:1115] Loading model weights took 14.2426 GB
INFO 07-13 17:03:51 worker.py:267] Memory profiling takes 5.27 seconds
INFO 07-13 17:03:51 worker.py:267] the current vLLM instance can use total_gpu_memory (23.64GiB) x gpu_memory_utilization (0.90) = 21.27GiB
INFO 07-13 17:03:51 worker.py:267] model weights take 14.24GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 4.35GiB; the rest of the memory reserved for KV Cache is 2.62GiB.
INFO 07-13 17:03:51 executor_base.py:111] # cuda blocks: 3069, # CPU blocks: 4681
INFO 07-13 17:03:51 executor_base.py:116] Maximum concurrency for 32768 tokens per request: 1.50x
INFO 07-13 17:03:56 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_ut

Capturing CUDA graph shapes: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:16<00:00,  2.10it/s]

INFO 07-13 17:04:12 model_runner.py:1562] Graph capturing finished in 17 secs, took 0.21 GiB
INFO 07-13 17:04:12 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 26.88 seconds


In [9]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate.from_template(template)

llm_chain = LLMChain(prompt=prompt, llm=raw_model)

question = "Who was the US president in the year the first Pokemon game was released?"

print(llm_chain.invoke(question))

/tmp/ipykernel_23255/1151242886.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(prompt=prompt, llm=raw_model)
Processed prompts: 100%|███████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.05it/s, est. speed input: 27.33 toks/s, output: 36.80 toks/s]

{'question': 'Who was the US president in the year the first Pokemon game was released?', 'text': ' The first Pokemon game was released in 1996. The US president in 1996 was Bill Clinton. Therefore, the answer is Bill Clinton.'}


In [2]:
from langchain_huggingface import HuggingFacePipeline

raw_model = HuggingFacePipeline.from_model_id(
    model_id="t-tech/T-lite-it-1.0",
    task="text-generation",
    dtype="float16",
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


OutOfMemoryError: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 243.56 MiB is free. Including non-PyTorch memory, this process has 23.40 GiB memory in use. Of the allocated memory 23.09 GiB is allocated by PyTorch, and 150.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
from langchain_huggingface import ChatHuggingFace

chat_model = ChatHuggingFace(llm=raw_model)

In [5]:
class_dataset[0], class_targets[0]

('hide new secretions from the parental units ', 0)

In [9]:
from pydantic import BaseModel, Field
class ResponseFormatter(BaseModel):
    """Always use this tool to structure your response to the user."""
    answer: str = Field(description="The answer to the user's question")
    followup_question: str = Field(description="A followup question the user could ask")

model_with_tools = llm.bind_tools([ResponseFormatter])
ai_msg = model_with_tools.invoke("What is the powerhouse of the cell?")

ConnectError: [Errno 111] Connection refused

In [34]:
ai_msg.tool_calls

[]

In [31]:
print(ai_msg.content)

<｜begin▁of▁sentence｜><｜User｜>Return a JSON object with key 'random_ints' and a value of 10 random ints in [0-99]<｜Assistant｜><think>
Okay, so the user wants a JSON object with a key 'random_ints' and a value that's a list of 10 random integers between 0 and 99. Hmm, let me think about how to approach this.

First, I need to generate 10 random numbers. I remember that in JavaScript, there's a Math.random() method which gives a number between 0 and 1. If I multiply that by 100, it'll give me a number between 0 and 100. But wait, that would include 100. So maybe I should use Math.floor() to round it down to get an integer between 0 and 99.

So the plan is to generate each number using Math.floor(Math.random() * 100), but I need to make sure that each number is unique. Oh, right, because if I just generate them one by one, there's a chance they might repeat. So perhaps I should create an array and add each number to it until it's filled up to 10 elements.

Let me outline the steps:
1. Init